# TrialGPT criterion matching — Kaggle GPU pipeline
Settings: **Accelerator = GPU T4 x2** (only GPU 0 is used), **Internet = On**.
Runs: data split → zero-shot baseline → LoRA fine-tuning (target 1.5B + draft 0.5B) → evaluation → KV / attention / speculative benchmarks. Total ≈ 40–50 min.
At the end, download `/kaggle/working/artifacts.zip`.

In [ ]:
REPO_URL = 'https://github.com/Abdullah8168/trialgpt-criterion-matching.git'
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!git clone -q $REPO_URL repo
os.chdir('repo')
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 pyarrow
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# B1/B2: fixed 37/8/8 patient-grouped split (seed 42), GPT-4 columns removed, leakage checks
!python src/prepare_data.py

In [ ]:
# B4 'before': zero-shot base model on validation
!python src/evaluate.py --split val --tag base

In [ ]:
# B3: LoRA fine-tuning of the target model (Qwen2.5-1.5B-Instruct)
!python src/train_lora.py

In [ ]:
# Model selection on validation only
!python src/evaluate.py --split val --tag finetuned --adapter outputs/adapter

In [ ]:
# Draft model for speculative decoding (Qwen2.5-0.5B-Instruct), same data/recipe
!python src/train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --out outputs/draft_adapter

In [ ]:
# FINAL held-out test evaluation (run once, after all choices were fixed on validation)
!python src/evaluate.py --split test --tag base
!python src/evaluate.py --split test --tag finetuned --adapter outputs/adapter

In [ ]:
# Task 3: KV cache vs no-cache
!python src/bench_kv.py --adapter outputs/adapter --repeats 3

In [ ]:
# Task 4: tiled online-softmax attention
!python src/bench_attention.py

In [ ]:
# Task 5: speculative decoding
!python src/bench_speculative.py --adapter outputs/adapter --draft_adapter outputs/draft_adapter --n_prompts 12 --ks 2,4,6

In [ ]:
!python src/make_figures.py
!rm -rf outputs/ckpt_*
!zip -qr /kaggle/working/artifacts.zip results data/split_manifest.json data/processed outputs
!ls -la /kaggle/working/artifacts.zip